# CMC 87708 — Secondary contextual source-boundary ablation (Gemma + Qwen, 1536)

Reproduces the revised-manuscript secondary experiment:
- Flow and Flow+Graph
- Basic and Bounded
- Gemma and Qwen
- 3 datasets × 500 targets
- 12,000 native outputs total
- detector state and traceability excluded from the LLM-facing contextual input
- deterministic decoding, max_new_tokens=1536

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ramita99/cmc_llm_87708.git
%cd cmc_llm_87708
!pip install -q -e .

In [ ]:
from google.colab import userdata
from huggingface_hub import login, hf_hub_download
hf_token=userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token,add_to_git_credential=False)

## Generate 6,000 contextual outputs per model

In [ ]:
!python scripts/06_run_contextual_ablation.py --config configs/contextual_final.yaml --model-key gemma --resume

In [ ]:
!python scripts/06_run_contextual_ablation.py --config configs/contextual_final.yaml --model-key qwen --resume

## Source-boundary metrics and paired binary statistics

In [ ]:
!python scripts/07_evaluate_contextual_ablation.py --config configs/contextual_final.yaml

## Contextual AlignScore

AlignScore is computed against the LLM-facing Flow or Flow+Graph context only.

In [ ]:
ckpt=hf_hub_download(repo_id="yzha/AlignScore",filename="AlignScore-large.ckpt",token=hf_token)
!python scripts/08_contextual_alignscore.py --config configs/contextual_final.yaml --checkpoint {ckpt} --batch-size 32 --device cuda